# 01 · Transcutaneous Electrical Stimulation

**Physics of Electrical Neurostimulation** · Taller Escuela de Neurociencia (FALAN School), Santiago, August 2026

Leonel Medina · Rodrigo Osorio · Cristian Morales — [NeuroEng@USACH](https://www.neuroeng-usach.cl), Universidad de Santiago de Chile

| | |
|---|---|
| Notebook 01 | **Transcutaneous stimulation** — the field, the axon, the strength-duration curve |
| Notebook 02 | **Interferential current** — two carriers, one beat, deep activation |
| Lab | TENS/EMS unit on your own forearm; handouts in `handouts/` |
| No install | `explorer/index.html` runs the two core figures in any browser |

### What you will be able to do by the end
1. Compute and read the electric field of a surface electrode pair over layered tissue, and say which feature of that field excites a nerve.
2. Predict where an axon fires from the activating function, then check that prediction against a real nonlinear membrane simulation.
3. Measure a strength-duration curve on your own forearm, fit rheobase and chronaxie, and compare all three: your data, this model, and the published human value.
4. Explain why cathodic and anodic thresholds differ — and the conditions under which they don't.

### How the 80 minutes are spent
| | |
|---|---|
| Module 0 — the electric field itself | ~20 min |
| Module 1 — one stimulus: does the fibre fire? | ~20 min |
| Module 2 — strength-duration curve, model vs. your forearm | ~30 min (simulation runs while you measure) |
| Wrap-up and hand over to Notebook 02 | ~10 min |

> **Start the Module 2 simulation early.** A full five-point sweep takes about a minute of real
> computation. Launch it, then go and put electrodes on someone's forearm while it runs.

## 0 · Setup

One cell, no editing. On Colab it also installs NEURON and PyFibers (about a minute).

In [1]:
# One-time setup: fetches the setup script if needed and runs it. On Colab it also
# installs NEURON, which requires the kernel to restart once -- if that happens,
# just run this cell again. Curious what it does? Open notebooks/setup_workshop.py
import pathlib, urllib.request
URL = ("https://raw.githubusercontent.com/neuroeng-usach/"
       "falan-neurostim-workshop/main/notebooks/setup_workshop.py")
if not pathlib.Path("setup_workshop.py").exists():
    urllib.request.urlretrieve(URL, "setup_workshop.py")
%run -i setup_workshop.py

RESTART REQUIRED — numpy changed from 2.3.3 to 2.2.6 while this kernel was running.
  Colab:  Runtime -> Restart session,  then run this cell again.
  Local:  restart the kernel, then run this cell again.
  Nothing below will work until you do; this is not an error in the
  notebook, it is how compiled Python packages behave.
repo root: /Users/leo/Library/CloudStorage/GoogleDrive-leonel.medina@usach.cl/My Drive/Outreach/FALAN School/release
[self-test] current conservation: integrated 0.9960 mA vs injected 1.0000 mA (ratio 0.9960) -- PASS
[self-test] all analytic sanity checks passed:
  continuity at boundary: 89.665828 vs 89.665778 mV
  homogeneous-limit match: 94.901672 vs 94.901672 mV
  insulating backing raises V: 807.8492 > 94.9017
  conductive backing lowers V: 10.1876 < 94.9017
[self-test] disc electrode checks passed:
  tiny-disc -> point-source convergence: rel diff 3.49e-06
  peak V monotonically decreasing with radius: ['3735.4', '2216.7', '889.5', '500.1', '272.0', '153.7']

--No graphics will be displayed.


ENVIRONMENT READY — field figures and axon simulations will both run.


True

## What is actually being computed here

Three modules, all in Python, no FEM license needed, and no hidden one-line formulas.

- **Module 0 — the electric field itself.** A two-layer volume conductor (skin+fat over muscle),
  derived from first principles by the method of images and validated against current
  conservation, in `src/layered_field.py`. The electrode can be a point source or a realistic
  **disc** of adjustable radius (a superposition of point sources), and the field is driven by a
  **bipolar pad pair** — source and sink, adjustable separation — matching how the TENS unit you
  will actually test with works.
- **Module 1 — cathodic vs. anodic, one stimulus at a time.** The *same* field from Module 0 now
  drives a real myelinated axon (MRG double-cable model, PyFibers/NEURON).
- **Module 2 — strength-duration curve.** Real bisection threshold searches on the nonlinear
  membrane, fitted for rheobase and chronaxie, compared against your own forearm measurements.

Every slider you move in Module 0 carries forward: Modules 1 and 2 inherit the tissue
parameters you set there, through the shared store `ws.STATE`. Print it any time to see the
full parameter set you are working with.

*No sliders in your environment?* Nothing breaks. Each interactive cell falls back to running
once with its defaults and tells you how to override them by hand — there is no separate
"fallback" notebook to keep track of.

## Module 0 — The electric field of a transcutaneous electrode pair

**Before you run it:** with the cathode (negative current) on and the muscle
layer *more* conductive than the shallow layer (the default), do you expect
the deep layer to concentrate current toward it, or spread it out? Run and check.

**Then try this:** raise the electrode radius from 0 (a point) to a realistic
disc pad (e.g. 5-10 mm, roughly a small TENS/EMS electrode). Does the peak
current density under the electrode go up or down? Does the activating
function at the fiber get sharper or more spread out?

**This is bipolar by default** -- two electrodes (a source and a sink)
`separation_mm` apart, exactly like the two pads on the TENS unit you'll
test with, instead of one electrode with an implicit distant return. Try
changing the separation: what happens to the field between the two pads
as they get closer together vs. farther apart?

### Step 1 — One interface, worked through completely

Everything in this workshop rests on one calculation, done once. Two media, one flat boundary.

**The physics is electrostatics, not electrodynamics.** Below about 10 kHz tissue behaves as a pure
resistor, so away from the electrode current is conserved and

$$\nabla \cdot (\sigma \nabla V_e) = 0.$$

No wave equation, no time. Time only enters later, through the membrane.

**A point source in an infinite uniform medium.** Inject $I_0$ at a point. Current spreads over
spheres of area $4\pi r^2$, so $J = I_0/4\pi r^2$; integrating $E = J/\sigma$ inward from infinity,

$$V_e(r) = \frac{I_0}{4\pi\sigma r}.$$

**The two conditions at any interface.** Put medium 1 ($\sigma_1$) in $z>0$, medium 2 ($\sigma_2$)
in $z<0$, boundary at $z=0$, and a point source at height $d$ inside medium 1:

| | condition | why |
|---|---|---|
| (i) | $V_e$ continuous | otherwise $E=-\nabla V_e$ is infinite |
| (ii) | $\sigma\,\partial V_e/\partial z$ continuous | otherwise charge piles up on the interface |

**The guess.** Write the answer as the real source plus *image* sources. The rule that makes this
legitimate is strict:

> In each region, you may only include terms whose singularity lies **outside** that region — except
> the real source, which must be inside.

A term $1/R$ is a solution of the equation everywhere except at its own singular point, so an image
placed outside the region adds no fictitious source where we are using the formula. Applying the
rule to each side:

| region | $1/R_+$, singular at $z=+d$ | $1/R_-$, singular at $z=-d$ | so the form is |
|---|---|---|---|
| medium 1, $z>0$ | **inside** — required, it is the electrode | outside — allowed as an image | $1/R_+ + K/R_-$ |
| medium 2, $z<0$ | outside — allowed as an image | **inside** — forbidden | $T/R_+$ |

$$V_1 = \frac{I_0}{4\pi\sigma_1}\left[\frac{1}{R_+} + \frac{K}{R_-}\right], \qquad
V_2 = \frac{I_0}{4\pi\sigma_1}\,\frac{T}{R_+}, \qquad R_\pm = \sqrt{\rho^2 + (z\mp d)^2}$$

**Why the two sides look different.** Medium 2 contains no electrode at all, so its field must be
produced entirely from outside it. Adding a $K'/R_-$ term there would assert a current source at
$z=-d$, several millimetres deep in tissue where nothing exists — and the formula would predict an
infinite potential at an ordinary point.

**Now solve.** On the plane $z=0$ the two distances coincide, $R_+ = R_- = \sqrt{\rho^2+d^2}$, and
the vertical derivatives are exactly opposite, $\partial_z(1/R_+) = +d/R^3$ and
$\partial_z(1/R_-) = -d/R^3$. So both conditions lose their $\rho$-dependence and become algebra:

$$\text{(i)}\quad 1 + K = T \qquad\qquad \text{(ii)}\quad \sigma_1 (1-K) = \sigma_2 T$$

$$\Rightarrow\quad \boxed{\;K = k = \frac{\sigma_1-\sigma_2}{\sigma_1+\sigma_2}\;}\qquad
\boxed{\;T = 1+k = \frac{2\sigma_1}{\sigma_1+\sigma_2}\;}$$

Two unknowns, two conditions, and — crucially — they hold at **every** point of the plane, not just
one. That is why the mirror position is forced rather than chosen: it is the only placement where a
single constant can satisfy a condition over a whole surface.

**Sanity checks.** $\sigma_2=\sigma_1 \Rightarrow k=0$: no interface, no image, back to the
infinite medium. $\sigma_2 > \sigma_1$ (muscle under fat) $\Rightarrow k<0$: a negative image, which
*lowers* the potential above and pulls current down into the better conductor — the behaviour you
will see in the field plot. $\sigma_2 \to 0$ (insulator below) $\Rightarrow k \to +1$: the boundary
becomes a perfect mirror. Hold on to that last one.

### Step 2 — The case that actually applies: the electrode sits *on* the boundary

Step 1 assumed the source floats at height $d$ above the interface. A TENS pad does not: it sits
directly on the skin, at $z=0$, **on the mirror itself**. That deserves care, because the reflection
map $z \mapsto -z$ has a fixed point exactly there — the image lands on top of the source.

**First, what the skin boundary is.** Air is an insulator, $\sigma_{\text{air}} = 0$, so no current
can cross it and condition (ii) becomes $\partial V_e/\partial z = 0$ at $z=0$. Put
$\sigma_2 = 0$ into the formula from Step 1:

$$k_{\text{air}} = \frac{\sigma_1 - 0}{\sigma_1 + 0} = +1$$

A **perfect mirror**: reflects everything, with no loss and no sign change.

**Now the degenerate geometry, done honestly.** Do not put the source on the boundary. Put it a
hair inside the skin, at $z = -\delta$ with $\delta>0$, where Step 1 applies unchanged, and then
take the limit:

$$\underbrace{\text{real source at } z=-\delta,\ \text{weight }1}_{\text{inside the tissue}} \;+\;
\underbrace{\text{image at } z=+\delta,\ \text{weight } k_{\text{air}}=+1}_{\text{in the air}}
\qquad \xrightarrow{\ \delta \to 0\ } \qquad \text{one source at } z=0 \text{ of weight } 2$$

The pair merges. The electrode and its own reflection become indistinguishable, and the strength
adds:

$$V_e(r) = 2\cdot\frac{I_0}{4\pi\sigma_1 r} = \boxed{\;\frac{I_0}{2\pi\sigma_1 r}\;}$$

**A surface electrode produces twice the potential a buried one would.** That is the whole origin of
the $2\pi$ you see throughout this notebook instead of the textbook $4\pi$: not a different formula,
just a source sitting on a perfect mirror. Physically it is obvious in hindsight — all the current
is forced into the half-space below instead of spreading in every direction.

**The same argument on an imperfect mirror.** If a source sat on the *deep* interface instead, where
the mirror coefficient is $k$ rather than $+1$, the merge would give weight $1+k$ — which is exactly
the transmission coefficient $T$ from Step 1. The perfect-mirror factor of 2 is just $T$ evaluated
at $\sigma_2 = 0$. One rule, two cases.

Run the next cell to confirm the doubling on the actual code.

In [ ]:
# Turn the deep contrast off (sigma2 = sigma1) so the skin is the only boundary left,
# then compare the code against the two textbook formulas from Step 2.
r_mm, z_mm, sigma = 5.0, -2.0, lf.DEFAULT_SIGMA1
R_m = np.hypot(r_mm, z_mm) * 1e-3                       # mm -> m

V_code = float(lf._point_potential(r_mm, z_mm, i0_mA=1.0, sigma1=sigma, sigma2=sigma))
V_surface = 1e-3 / (2 * np.pi * sigma * R_m) * 1e3      # I0/(2*pi*sigma*R), in mV
V_buried  = 1e-3 / (4 * np.pi * sigma * R_m) * 1e3      # I0/(4*pi*sigma*R), in mV

print(f"layered_field.py        {V_code:9.4f} mV")
print(f"I0/(2*pi*sigma*R)       {V_surface:9.4f} mV   <- electrode ON the skin")
print(f"I0/(4*pi*sigma*R)       {V_buried:9.4f} mV   <- same source buried in bulk tissue")
print(f"ratio                   {V_code / V_buried:9.4f}   <- the mirror doubling")

### Step 3 — Two interfaces, and why one image is no longer enough

Your forearm has a second boundary: skin+fat over muscle at depth $h$. Now there are two mirrors
facing each other — the perfect one at the skin ($+1$) and the partial one at depth ($k$).

Place an image to satisfy the deep interface and it violates the air boundary. Patch the air
boundary and the new image violates the deep interface. Neither patch is free, and the process never
terminates. This is the optical hall of mirrors, and it is why **a single image is not an
approximation here, it is simply wrong**.

Tracking the reflections (each mirroring sends $z \mapsto 2a-z$) gives images at multiples of $2h$,
picking up one factor of $k$ per bounce off the deep interface and nothing at the perfect skin
mirror:

$$V_1(r,z) = \frac{I_0}{2\pi\sigma_1}\left[\frac{1}{R_0} + \sum_{n=1}^{\infty} k^n\left(\frac{1}{R_n^{+}} + \frac{1}{R_n^{-}}\right)\right], \qquad R_n^{\pm}=\sqrt{r^2+(z\mp 2nh)^2}$$

$$V_2(r,z) = \frac{I_0}{\pi(\sigma_1+\sigma_2)}\sum_{n=0}^{\infty} \frac{k^n}{R'_n}, \qquad R'_n=\sqrt{r^2+(2nh-z)^2}$$

$V_1$ is the potential in skin+fat, $V_2$ in the muscle where the nerve lives. Note the prefactors:
$V_1$ carries the $2\pi$ of Step 2's mirror doubling, and $V_2$ carries
$\frac{I_0}{2\pi\sigma_1}\,(1+k) = \frac{I_0}{\pi(\sigma_1+\sigma_2)}$ — the doubling *and* one
transmission through the deep interface. Every symbol traces back to Steps 1 and 2.

Since $|k|<1$ always (both conductivities are positive), the series converges geometrically. The
figure below shows the ladder and measures how fast.

**Why the checks are not decoration.** An earlier version of `layered_field.py` used a single image
of weight $k$ — correct for one interface, wrong for two. It produced perfectly plausible field
plots and was off by about 60%, because the true correction is the whole series
$2\sum_{n\ge1}k^n = 2k/(1-k)$ rather than its first term $2k$. What caught it was integrating the
current through a deep plane and comparing with what was injected. That check ran in your setup
cell.

**Two shortcuts we now get for free**, because the equation is linear in $I_0$:

- a **disc** electrode of radius $a$ is many point sources sharing the current,
  $V_{\text{disc}} = \sum_i V_{\text{point}}(|\mathbf{r}-\mathbf{r}_i|; I_0/N)$;
- a **bipolar pad pair** is a source and a sink, $V(x+\tfrac{s}{2}) - V(x-\tfrac{s}{2})$. No
  separate ground, exactly like a real TENS unit.

*(Honest caveat: the disc assumes uniform current density across the pad. A real gelled electrode
behaves more like a constant-potential surface, which crowds current at the pad's edge. Every
electrode-size effect below has the right direction, not necessarily the exact magnitude.)*

In [ ]:
# The image construction, and how quickly the series converges.
# Change the conductivities to see k -- and the whole ladder -- change.
tp.draw_image_construction(sigma1=lf.DEFAULT_SIGMA1, sigma2=lf.DEFAULT_SIGMA2,
                           h_mm=lf.DEFAULT_H_MM)

### From field to excitation: the cable equation

$V_e$ is a property of the *tissue*. It exists whether or not a nerve is there. So what does a fibre
sitting in that field actually feel?

**The setup.** Treat the axon as a thin cable: an intracellular core of axial resistance $r_i$ per
unit length, separated from the extracellular space by a membrane with capacitance $c_m$ and ionic
current $i_{\text{ion}}$ per unit length. Two potentials matter — $V_i$ inside, $V_e$ outside — and
the membrane only responds to the difference, $V_m = V_i - V_e$.

**Three steps.** Ohm's law along the core:

$$i_a = -\frac{1}{r_i}\frac{\partial V_i}{\partial x}$$

Current conservation — whatever leaves the core axially must cross the membrane:

$$i_m = -\frac{\partial i_a}{\partial x} = \frac{1}{r_i}\frac{\partial^2 V_i}{\partial x^2}$$

And the membrane itself: $i_m = c_m \dfrac{\partial V_m}{\partial t} + i_{\text{ion}}$.

**The substitution that does all the work.** Write $V_i = V_m + V_e$:

$$c_m\frac{\partial V_m}{\partial t} + i_{\text{ion}} \;=\; \frac{1}{r_i}\frac{\partial^2 V_m}{\partial x^2} \;+\; \underbrace{\frac{1}{r_i}\frac{\partial^2 V_e}{\partial x^2}}_{\textstyle f(x)}$$

Everything on the left is the fibre's own business. The external field appears in exactly **one**
place, as a source term driving the cable — and only through its **second derivative along the
fibre**. That term is the **activating function** (Rattay, 1986):

$$\boxed{\;f(x) = \frac{1}{r_i}\frac{\partial^2 V_e}{\partial x^2}\;}$$

**Reading it.** $f > 0$ injects current into the fibre: **depolarizing**. $f < 0$ removes it:
**hyperpolarizing**. Because a stimulus that depolarizes somewhere must hyperpolarize elsewhere
(the second derivative of a bounded function integrates to zero), every cathode comes with flanking
*virtual anodes* — which is why the plot below always has both colours.

**Three consequences worth stating out loud.**

1. **Magnitude is not what excites.** A large but *uniform* $V_e$ has zero second derivative and
   does nothing at all. Curvature excites, not amplitude.
2. **For a myelinated fibre the derivative becomes a difference** across nodes of Ranvier:
   $f_n \propto V_{e,n-1} - 2V_{e,n} + V_{e,n+1}$. Internodal distance therefore sets which fibres
   are recruited first — the reason large fibres, with their widely spaced nodes, go first.
3. **It is a linear approximation** — a passive prediction about a nonlinear membrane. It should be
   most accurate right at threshold, where the membrane has barely moved from rest, and should
   degrade well above it. **Module 1 tests exactly that**, by comparing this prediction against
   where a full nonlinear NEURON simulation actually initiates the spike.

In [ ]:
panel0 = ws.Panel(
    tp.draw_module0,
    controls=[
        ws.num("sigma1",              "sigma1 skin+fat (S/m)",        0.01, 0.30, 0.01, lf.DEFAULT_SIGMA1),
        ws.num("sigma2",              "sigma2 muscle (S/m)",          0.05, 1.00, 0.01, lf.DEFAULT_SIGMA2),
        ws.num("h_mm",                "shallow layer thickness (mm)", 1.0, 12.0,  0.5,  lf.DEFAULT_H_MM),
        ws.num("i0_mA",               "current (mA, neg = cathodic)", -10.0, 10.0, 0.1, -2.0),
        ws.num("fiber_depth_mm",      "fibre depth (mm)",             1.0, 20.0,  0.5,  8.0),
        ws.num("electrode_radius_mm", "pad radius (mm, 0 = point)",   0.0, 15.0,  0.5,  4.0),
        ws.num("separation_mm",       "pad separation (mm)",          20.0, 120.0, 5.0, 60.0),
    ],
    button="Draw field",
).show()

## Module 1 — One stimulus: does the fiber fire? (cathodic vs. anodic)

Same field as Module 0, now driving a real fiber. Pick an amplitude, pulse
width, and polarity, then run -- the top panel shows the activating-function
prediction from Module 0's theory (dotted line) next to where the real,
nonlinear NEURON simulation actually initiates the spike, if it fires
(dashed line).

**Electrode mode defaults to monopolar** (single electrode, implicit distant
return) -- this is what produces the classic, literature-matching
cathodic/anodic threshold asymmetry (~3.9x) you'll read about below.

**Try switching to bipolar mode with the default (symmetric) settings.**
What happens to the cathodic/anodic difference? Think about *why* before
reading the facilitator notes -- it's a real, physically meaningful result
(not a bug), and it says something important about what "reversing
polarity" actually means once there's a real return pad instead of a
distant implicit one. Bipolar mode also needs a longer simulated fiber and
runs noticeably slower (~15-25s per threshold vs. ~5-10s monopolar) --
that's expected, not a freeze.

### From prediction to test: why run a real simulation at all?

The activating function (Module 0) is a *linear* approximation -- it assumes the membrane responds proportionally to the driving field, which only holds for small perturbations away from rest. Whether the fiber actually fires a propagating action potential depends on the full *nonlinear* membrane dynamics (voltage-gated sodium/potassium channels -- the same Hodgkin-Huxley-family kinetics built into the MRG model here), which is exactly what PyFibers/NEURON simulates directly, compartment by compartment, with no shortcuts.

The top panel below shows both: the **linear prediction** (peak of the activating function, dotted line) and, if the fiber fires, **where the nonlinear simulation actually initiates the action potential** (dashed line -- the earliest node to cross threshold). Near threshold amplitude they should agree closely, since the linear approximation is most accurate right where the membrane is just barely being pushed over the edge -- that's exactly the regime it was derived for.

**Try this:** run near threshold first, then crank the amplitude well past it. Does the actual initiation site stay locked to the prediction, or drift as the response becomes more nonlinear?

In [ ]:
# Tissue geometry is inherited from Module 0 -- change it there and it changes here.
panel1 = ws.Panel(
    tp.draw_module1,
    controls=[
        ws.choice("diameter",       "fibre diameter (um)",
                  [5.7, 7.3, 8.7, 10.0, 11.5, 12.8, 14.0, 15.0, 16.0], 10.0),
        ws.num("amplitude_mA",     "amplitude (mA)",    0.05, 20.0, 0.05, 3.0),
        ws.num("pulse_width_ms",   "pulse width (ms)",  0.02,  2.0, 0.01, 0.3),
        ws.choice("polarity",      "polarity", ["cathodic", "anodic"], "cathodic"),
        ws.choice("electrode_mode", "electrode mode",
                  [("monopolar (single electrode)", "monopolar"),
                   ("bipolar (pad pair, like the real TENS)", "bipolar")], "monopolar"),
    ],
    inherit=["fiber_depth_mm", "sigma1", "sigma2", "h_mm",
             "electrode_radius_mm", "separation_mm"],
    button="Run simulation",
    note="Bipolar mode uses a longer fibre and takes ~15-25 s per run; monopolar ~5-10 s.",
).show()

## Module 2 — Strength-duration curve

Computes threshold at several pulse widths (real nonlinear simulations — can
take up to ~1-2 minutes, longer in bipolar mode), fits rheobase & chronaxie,
and compares to the published human ulnar-nerve values. Then compare to your
own forearm data.

Uses the same electrode mode / pad separation as Module 1 above (the
`mode1_dd` / `sep1_sl` widgets) -- change those first if you want a bipolar
sweep here.

### Which pulse widths to sweep

The model is happy over 50 us - 1 ms. Your **TENS Ultima Neo in monophasic rectangular mode is
limited to 50-250 us**, so the two ranges only partly overlap. Sweep the device-matched set if
you want a like-for-like comparison, or the wider set to see the whole curve bend. Doing both and
discussing the mismatch is the honest option — and it is the range where chronaxie actually lives.

Set `MY_DATA` to whatever you measured on the forearm (one `pulse_width_ms, threshold_mA` pair per
line; paste straight from the handout table, comments and all — non-numeric lines are ignored).

In [ ]:
# --- what to sweep ----------------------------------------------------------
PULSE_WIDTHS_MS = (0.05, 0.10, 0.20, 0.50, 1.00)      # full model range
# PULSE_WIDTHS_MS = (0.05, 0.10, 0.15, 0.20, 0.25)    # Ultima Neo monophasic range
if ws.FAST:                                            # smoke test only (WORKSHOP_FAST=1)
    PULSE_WIDTHS_MS = PULSE_WIDTHS_MS[::2]

# --- your forearm measurements (pulse_width_ms, motor threshold mA) ---------
MY_DATA = """
0.05,
0.10,
0.15,
0.20,
0.25,
"""

sim = tp.compute_module2(
    diameter=ws.STATE["diameter"], fiber_depth_mm=ws.STATE["fiber_depth_mm"],
    sigma1=ws.STATE["sigma1"], sigma2=ws.STATE["sigma2"], h_mm=ws.STATE["h_mm"],
    polarity=ws.STATE["polarity"], pulse_widths_ms=PULSE_WIDTHS_MS,
    electrode_radius_mm=ws.STATE["electrode_radius_mm"],
    electrode_mode=ws.STATE["electrode_mode"], separation_mm=ws.STATE["separation_mm"],
)
pts, fit = sim

my_pts = ws.parse_pairs(MY_DATA)
tp.plot_module2(pts, fit, my_pts or None, weiss.fit_weiss(my_pts) if len(my_pts) > 1 else None)

## Bridge to the lab — measuring this on a forearm

The number this notebook predicts is a **motor threshold as a function of pulse width**, so the
measurement has to be a motor threshold too. The full protocol is in
`handouts/01_transcutaneous_handout.md`; the four things that decide whether your curve has any
shape at all:

1. **Single pulses, not trains.** At 30-50 Hz the contraction appears through mechanical fusion
   and temporal summation, and that threshold is almost completely insensitive to pulse width —
   a flat "curve" is the guaranteed result. Use the lowest frequency the device offers (1-2 Hz).
2. **Monophasic mode.** Symmetric biphasic pulses reverse polarity within each pulse, which both
   erases the cathodic/anodic comparison of Module 1 and compresses the long-pulse-width end of
   the curve. Confirm which lead is the cathode before you start — do not trust a red/black
   convention.
3. **A visible, repeatable endpoint.** Median nerve at the wrist, watching thumb abduction, with a
   light pointer taped to the thumb against a ruler. Threshold = the lowest amplitude giving a
   visible response in **5 of 10 pulses**.
4. **Randomised pulse-width order, and a blind scorer.** Then repeat the first condition at the
   end: skin impedance drifts downward over a session by about as much as the effect you are
   trying to measure.

Your instructors have bench-tested the unit you are using for the traps that produce a
plausible-looking but meaningless curve: output-compliance clipping, trains where you expected
single pulses, and which physical lead is actually the cathode. Ask them what they found before you
start — the answers should be on the whiteboard.

## What to compare, and what it means

| Quantity | Your forearm | This model | Published (near-nerve ulnar) |
|---|---|---|---|
| Rheobase | | | 0.91 mA (SD 0.37) |
| Chronaxie | | | 0.32 ms (SD 0.17) |

**Chronaxie is the robust prediction.** It mostly reflects membrane kinetics, so the model lands
near the published value almost regardless of the depth and conductivity you chose.
**Rheobase is the sensitive one** — it scales with how far the electrode sits from the nerve and
with the tissue conductivities. If your measured rheobase does not match, go back to Module 0 and
adjust `sigma1`, `sigma2` and `fiber_depth_mm` until it does, then ask what that implies about
where the nerve actually is under your pads. A transcutaneous surface measurement sits farther
from the nerve than Tsui et al.'s near-nerve needle, so a **higher** rheobase is the expected result, not an
error.

### Next: Notebook 02
Everything so far used one low-frequency pulse through one pad pair, and its reach into depth was
set by pad separation. Notebook 02 asks the opposite question: can you deliver a low-frequency
stimulus *deep* while keeping the skin comfortable, by generating the low frequency **inside** the
tissue instead of applying it at the surface? Same volume conductor, four pads.